In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import os

In [2]:
SCRIPT_DIR = Path(os.getcwd()).resolve()

ROOT_DIR = SCRIPT_DIR.parent

DATA_DIR = ROOT_DIR / "data" / "raw"
DATA_CLEAN_DIR = ROOT_DIR / "data" / "clean"

## all matches csv

In [3]:
df = pd.read_csv(DATA_DIR / 'all_matches.csv')
df.tail(1)

,date,home_team,away_team,home_score,away_score,tournament,country,neutral
51645,2026-06-11,Mexico,South Africa,2,0,World Cup,Mexico,False


In [4]:
# lier le nom des pays
df["country"] = df["country"].str.replace(" ", "_")
df["home_team"] = df["home_team"].str.replace(" ", "_")
df["away_team"] = df["away_team"].str.replace(" ", "_")

In [5]:
# Nettoyage de base
df["date"] = pd.to_datetime(df["date"], errors="coerce")

# Colonnes texte
text_cols = ["home_team", "away_team", "tournament", "country"]
for col in text_cols:
    df[col] = (
        df[col]
        .astype("string")
        .str.normalize("NFKC")
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
    )

# Booléen neutral
df["neutral"] = (
    df["neutral"]
    .astype(str)
    .str.lower()
    .map({"true": True, "false": False, "1": True, "0": False})
)

# Scores
df["home_score"] = pd.to_numeric(df["home_score"], errors="coerce").astype("Int64")
df["away_score"] = pd.to_numeric(df["away_score"], errors="coerce").astype("Int64")

In [6]:
# Suppression des lignes invalides
df = df.dropna(subset=["date", "home_team", "away_team", "home_score", "away_score", "tournament", "country", "neutral"])

# Sécurité minimale
df = df[(df["home_score"] >= 0) & (df["away_score"] >= 0)]
df = df[df["home_team"] != df["away_team"]]

# Doublons exacts
df = df.drop_duplicates(
    subset=["date", "home_team", "away_team", "home_score", "away_score", "tournament", "country", "neutral"]
)

# Tri chronologique
df = df.sort_values(["date", "home_team", "away_team"]).reset_index(drop=True)

In [7]:
df["goal_diff"] = df["home_score"] - df["away_score"]
df["total_goals"] = df["home_score"] + df["away_score"]

df["result"] = np.select(
    [
        df["goal_diff"] > 0,
        df["goal_diff"] == 0,
        df["goal_diff"] < 0,
    ],
    [
        "home_win",
        "draw",
        "away_win",
    ],
    default=np.nan,
)

# Optionnel : indicateur match à terrain neutre
df["is_neutral"] = df["neutral"].astype("int8")

In [8]:
df.to_parquet(DATA_CLEAN_DIR / "all_matches_clean_full.parquet", index=False)

In [9]:
df.tail()

,date,home_team,away_team,home_score,away_score,tournament,country,neutral,goal_diff,total_goals,result,is_neutral
51641,2026-06-10,Algeria,Bolivia,4,0,Friendly,United_States,True,4,4,home_win,1
51642,2026-06-10,England,Costa_Rica,3,0,Friendly,United_States,True,3,3,home_win,1
51643,2026-06-10,Pakistan,Afghanistan,2,0,Diamond Jubilee Tournament,Maldives,True,2,2,home_win,1
51644,2026-06-10,Portugal,Nigeria,2,1,Friendly,Portugal,False,1,3,home_win,0
51645,2026-06-11,Mexico,South_Africa,2,0,World Cup,Mexico,False,2,2,home_win,0


In [10]:
np.sort(df['country'].unique())

array(['Afghanistan', 'Albania', 'Algeria', 'Andorra', 'Angola',
       'Anguilla', 'Antigua_and_Barbuda', 'Argentina', 'Armenia', 'Aruba',
       'Australia', 'Austria', 'Azerbaijan', 'Bahamas', 'Bahrain',
       'Bangladesh', 'Barbados', 'Belarus', 'Belgian_Congo', 'Belgium',
       'Belize', 'Benin', 'Bermuda', 'Bhutan', 'Bohemia',
       'Bohemia_and_Moravia', 'Bolivia', 'Bonaire',
       'Bosnia_and_Herzegovina', 'Botswana', 'Brazil', 'British_Guiana',
       'British_Honduras', 'British_Virgin_Islands', 'Brunei', 'Bulgaria',
       'Burkina_Faso', 'Burma', 'Burundi', 'CIS', 'Cambodia', 'Cameroon',
       'Canada', 'Cape_Verde', 'Cayman_Islands',
       'Central_African_Republic', 'Ceylon', 'Chad', 'Chile', 'China',
       'Colombia', 'Comoros', 'Congo', 'Congo-Leopoldville',
       'Cook_Islands', 'Costa_Rica', 'Croatia', 'Cuba', 'Curaçao',
       'Cyprus', 'Czechia', 'Czechoslovakia', 'DR_Congo', 'Dahomey',
       'Dem_Rep_of_the_Congo', 'Denmark', 'Djibouti', 'Dominica',
      

## male palyers

In [11]:
usecols = [
    "player_id",
    "short_name",
    "overall",
    "potential",
    "value_eur",
    "wage_eur",
    "age",
    "height_cm",
    "weight_kg",
    "club_name",
    "league_name",
    "club_position",
    "nationality_name",
    "nation_team_id",
    "nation_position",
    "preferred_foot",
    "weak_foot",
    "skill_moves",
    "international_reputation",
    "pace",
    "shooting",
    "passing",
    "dribbling",
    "defending",
    "physic",
]

df = pd.read_csv(
    DATA_DIR / "male_players.csv",
    usecols=usecols,
    low_memory=False
)

In [12]:
# Nettoyage texte
text_cols = [
    "short_name", "club_name", "league_name",
    "club_position", "nationality_name", "nation_position", "preferred_foot",
]

for col in text_cols:
    df[col] = (
        df[col]
        .astype("string")
        .str.normalize("NFKC")
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
    )


# Conversion numérique
num_cols = [
    "overall", "potential", "value_eur", "wage_eur", "age", "height_cm", "weight_kg",
    "weak_foot", "skill_moves", "international_reputation",
    "pace", "shooting", "passing", "dribbling", "defending", "physic",
    "nation_team_id"
]

for col in num_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Si nation_team_id doit rester entier mais avec des NA
df["nation_team_id"] = df["nation_team_id"].astype("Int64")
df["player_id"] = pd.to_numeric(df["player_id"], errors="coerce").astype("Int64")

In [13]:
# Suppression des lignes inutilisables
df = df.dropna(subset=["player_id", "short_name", "overall"])

# Quelques contraintes de cohérence
df = df[(df["overall"] >= 0) & (df["overall"] <= 99)]
df = df[(df["potential"].isna()) | ((df["potential"] >= 0) & (df["potential"] <= 99))]
df = df[(df["age"].isna()) | ((df["age"] >= 14) & (df["age"] <= 50))]
df = df[(df["height_cm"].isna()) | ((df["height_cm"] >= 140) & (df["height_cm"] <= 230))]
df = df[(df["weight_kg"].isna()) | ((df["weight_kg"] >= 40) & (df["weight_kg"] <= 140))]


# Déduplication
# On garde une ligne par joueur si le dataset contient plusieurs versions
# Priorité à la version la plus récente et/ou la plus forte
sort_cols = ["player_id", "overall", "potential"]
df = df.sort_values(sort_cols, ascending=[True, False, False])

df = df.drop_duplicates(subset=["player_id"], keep="first").reset_index(drop=True)

In [14]:
# Sauvegarde
df.to_parquet(DATA_CLEAN_DIR / "players_clean.parquet", index=False)

In [15]:
df.head()

,player_id,short_name,overall,potential,value_eur,wage_eur,age,height_cm,weight_kg,club_name,...,preferred_foot,weak_foot,skill_moves,international_reputation,pace,shooting,passing,dribbling,defending,physic
0,2,G. Pasquale,69,69,625000.0,7000.0,33,182,72,Udinese,...,Left,3,2,2,71.0,59.0,69.0,70.0,71.0,71.0
1,11,R. Rocchi,68,68,675000.0,8000.0,32,183,75,Metz,...,Right,3,3,1,52.0,62.0,69.0,68.0,56.0,69.0
2,16,Luis García,70,70,450000.0,6000.0,37,178,65,AS Eupen,...,Right,4,3,1,53.0,68.0,73.0,69.0,57.0,63.0
3,27,J. Cole,74,74,2400000.0,35000.0,32,176,73,Aston Villa,...,Right,4,4,2,60.0,70.0,78.0,80.0,36.0,55.0
4,28,Manu Herrera,76,76,4300000.0,45000.0,32,180,75,Elche,...,Left,3,1,1,NaN,NaN,NaN,NaN,NaN,NaN


In [16]:
df.columns

Index(['player_id', 'short_name', 'overall', 'potential', 'value_eur',
       'wage_eur', 'age', 'height_cm', 'weight_kg', 'club_name', 'league_name',
       'club_position', 'nationality_name', 'nation_team_id',
       'nation_position', 'preferred_foot', 'weak_foot', 'skill_moves',
       'international_reputation', 'pace', 'shooting', 'passing', 'dribbling',
       'defending', 'physic'],
      dtype='object')

In [17]:
df.dtypes

player_id                            Int64
short_name                  string[python]
overall                              int64
potential                            int64
value_eur                          float64
wage_eur                           float64
age                                  int64
height_cm                            int64
weight_kg                            int64
club_name                   string[python]
league_name                 string[python]
club_position               string[python]
nationality_name            string[python]
nation_team_id                       Int64
nation_position             string[python]
preferred_foot              string[python]
weak_foot                            int64
skill_moves                          int64
international_reputation             int64
pace                               float64
shooting                           float64
passing                            float64
dribbling                          float64
defending  

## create national team strength

In [18]:
# on repart avec le csv clean player
df = df.copy()

df = df[df["nation_team_id"].notna()].copy()

# Sécurité
df["nation_team_id"] = df["nation_team_id"].astype("Int64")
# df["nationality_name"] = (
#     df["nationality_name"]
#     .astype("string")
#     .str.normalize("NFKC")
#     .str.strip()
#     .str.replace(r"\s+", " ", regex=True)
# )

In [19]:
# utils
def safe_mean(s):
    s = pd.to_numeric(s, errors="coerce")
    return s.mean()

def safe_sum(s):
    s = pd.to_numeric(s, errors="coerce")
    return s.sum()

def top_n(df_group, n=11):
    # Tri par overall décroissant, puis potential décroissant
    return df_group.sort_values(
        ["overall", "potential", "age"],
        ascending=[False, False, True]
    ).head(n)

In [20]:
# Colonnes numériques
num_cols = [
    "overall", "potential", "value_eur", "wage_eur", "age",
    "height_cm", "weight_kg", "weak_foot", "skill_moves",
    "international_reputation", "pace", "shooting", "passing",
    "dribbling", "defending", "physic"
]

for col in num_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")


df["is_80_plus"] = (df["overall"] >= 80).astype("int8")
df["is_85_plus"] = (df["overall"] >= 85).astype("int8")
df["is_90_plus"] = (df["overall"] >= 90).astype("int8")

In [21]:
# Agrégation par nation
rows = []

for (nation_team_id, nationality_name), g in df.groupby(["nation_team_id", "nationality_name"], dropna=False):
    g = g.copy()
    g = g.sort_values(["overall", "potential", "age"], ascending=[False, False, True])

    top5 = top_n(g, 5)
    top11 = top_n(g, 11)
    top23 = top_n(g, 23)


    # Valeurs robustes
    row = {
        "nation_team_id": nation_team_id,
        "nationality_name": nationality_name,
        "players_in_dataset": len(g),
        "players_top23_available": len(top23),

        # Niveau global
        "overall_mean_all": g["overall"].mean(),
        "overall_median_all": g["overall"].median(),
        "overall_max_all": g["overall"].max(),
        "potential_mean_all": g["potential"].mean(),

        # Top 5
        "top5_overall_mean": top5["overall"].mean(),
        "top5_overall_min": top5["overall"].min(),
        "top5_overall_max": top5["overall"].max(),
        "top5_value_sum_eur": top5["value_eur"].sum(),
        "top5_age_mean": top5["age"].mean(),

        # Top 11
        "top11_overall_mean": top11["overall"].mean(),
        "top11_overall_std": top11["overall"].std(),
        "top11_overall_min": top11["overall"].min(),
        "top11_overall_max": top11["overall"].max(),
        "top11_potential_mean": top11["potential"].mean(),
        "top11_age_mean": top11["age"].mean(),
        "top11_height_mean": top11["height_cm"].mean(),
        "top11_weight_mean": top11["weight_kg"].mean(),
        "top11_value_sum_eur": top11["value_eur"].sum(),
        "top11_wage_sum_eur": top11["wage_eur"].sum(),

        # Capacités techniques top 11
        "top11_pace_mean": top11["pace"].mean(),
        "top11_shooting_mean": top11["shooting"].mean(),
        "top11_passing_mean": top11["passing"].mean(),
        "top11_dribbling_mean": top11["dribbling"].mean(),
        "top11_defending_mean": top11["defending"].mean(),
        "top11_physic_mean": top11["physic"].mean(),

        # Top 23
        "top23_overall_mean": top23["overall"].mean(),
        "top23_overall_std": top23["overall"].std(),
        "top23_overall_min": top23["overall"].min(),
        "top23_overall_max": top23["overall"].max(),
        "top23_potential_mean": top23["potential"].mean(),
        "top23_age_mean": top23["age"].mean(),
        "top23_age_std": top23["age"].std(),
        "top23_value_sum_eur": top23["value_eur"].sum(),
        "top23_wage_sum_eur": top23["wage_eur"].sum(),

        # Profondeur d'effectif
        "count_80_plus_all": int(g["is_80_plus"].sum()),
        "count_85_plus_all": int(g["is_85_plus"].sum()),
        "count_90_plus_all": int(g["is_90_plus"].sum()),
        "count_80_plus_top23": int(top23["is_80_plus"].sum()),
        "count_85_plus_top23": int(top23["is_85_plus"].sum()),
        "count_90_plus_top23": int(top23["is_90_plus"].sum()),

        # Médiane / robustesse
        "overall_iqr_all": g["overall"].quantile(0.75) - g["overall"].quantile(0.25),
        "overall_iqr_top23": top23["overall"].quantile(0.75) - top23["overall"].quantile(0.25),
    }

    # Score synthétique simple, utile comme feature additionnelle
    # (tu pourras le recalibrer plus tard)
    score = (
        0.45 * row["top11_overall_mean"] +
        0.20 * row["top23_overall_mean"] +
        0.15 * row["top11_passing_mean"] +
        0.10 * row["top11_defending_mean"] +
        0.10 * row["top11_shooting_mean"]
    )
    row["team_strength_score"] = score

    rows.append(row)

teams_strength = pd.DataFrame(rows)

In [22]:
# Nettoyage final

# Remplace les colonnes entièrement vides par 0 si besoin
numeric_cols = teams_strength.select_dtypes(include=[np.number]).columns
teams_strength[numeric_cols] = teams_strength[numeric_cols].replace([np.inf, -np.inf], np.nan)

# Pour les features manquantes liées à petits effectifs, on garde les NaN
# mais on peut remplir certains comptes à 0
for col in ["top23_gk_count", "top23_att_count", "top23_mid_count", "top23_def_count",
            "count_80_plus_all", "count_85_plus_all", "count_90_plus_all",
            "count_80_plus_top23", "count_85_plus_top23", "count_90_plus_top23"]:
    if col in teams_strength.columns:
        teams_strength[col] = teams_strength[col].fillna(0).astype("int64")

# Tri optionnel
teams_strength = teams_strength.sort_values(
    ["team_strength_score", "top11_overall_mean", "top23_overall_mean"],
    ascending=False
).reset_index(drop=True)

# chengement des noms:
teams_strength['nationality_name'] = teams_strength['nationality_name'].str.strip().str.replace(r'\s+', '_', regex=True)

# Définition des correspondances {ancien_nom: nouveau_nom}
changements = {
    'China_PR': 'China',
    "Côte_d'Ivoire": 'Ivory_Coast',
    "Czech_Republic": 'Czechia',
}

# Application du remplacement
teams_strength['nationality_name'] = teams_strength['nationality_name'].replace(changements)

In [23]:
teams_strength.to_parquet(DATA_CLEAN_DIR / "national_teams_strength.parquet", index=False)

In [24]:
teams_strength.head()

,nation_team_id,nationality_name,players_in_dataset,players_top23_available,overall_mean_all,overall_median_all,overall_max_all,potential_mean_all,top5_overall_mean,top5_overall_min,...,top23_wage_sum_eur,count_80_plus_all,count_85_plus_all,count_90_plus_all,count_80_plus_top23,count_85_plus_top23,count_90_plus_top23,overall_iqr_all,overall_iqr_top23,team_strength_score
0,1337,Germany,54,23,83.555556,84.0,92,86.500000,90.2,89,...,3767000.0,45,24,4,23,23,4,5.0,3.5,84.043478
1,1362,Spain,53,23,83.754717,84.0,91,86.943396,89.8,89,...,4199000.0,49,19,2,23,19,2,4.0,3.5,84.005375
2,1335,France,53,23,83.735849,84.0,91,86.981132,89.8,88,...,3740000.0,48,20,3,23,20,3,5.0,2.5,82.192117
3,1318,England,55,23,82.600000,83.0,90,85.127273,87.6,86,...,3537000.0,48,15,1,23,15,1,5.0,2.0,81.946838
4,1354,Portugal,53,23,82.150943,82.0,94,84.415094,89.4,88,...,3127000.0,38,9,1,23,9,1,5.0,3.0,81.651383


In [25]:
teams_strength.columns

Index(['nation_team_id', 'nationality_name', 'players_in_dataset',
       'players_top23_available', 'overall_mean_all', 'overall_median_all',
       'overall_max_all', 'potential_mean_all', 'top5_overall_mean',
       'top5_overall_min', 'top5_overall_max', 'top5_value_sum_eur',
       'top5_age_mean', 'top11_overall_mean', 'top11_overall_std',
       'top11_overall_min', 'top11_overall_max', 'top11_potential_mean',
       'top11_age_mean', 'top11_height_mean', 'top11_weight_mean',
       'top11_value_sum_eur', 'top11_wage_sum_eur', 'top11_pace_mean',
       'top11_shooting_mean', 'top11_passing_mean', 'top11_dribbling_mean',
       'top11_defending_mean', 'top11_physic_mean', 'top23_overall_mean',
       'top23_overall_std', 'top23_overall_min', 'top23_overall_max',
       'top23_potential_mean', 'top23_age_mean', 'top23_age_std',
       'top23_value_sum_eur', 'top23_wage_sum_eur', 'count_80_plus_all',
       'count_85_plus_all', 'count_90_plus_all', 'count_80_plus_top23',
       'co

In [26]:
teams_strength.dtypes

nation_team_id               int64
nationality_name            object
players_in_dataset           int64
players_top23_available      int64
overall_mean_all           float64
overall_median_all         float64
overall_max_all              int64
potential_mean_all         float64
top5_overall_mean          float64
top5_overall_min             int64
top5_overall_max             int64
top5_value_sum_eur         float64
top5_age_mean              float64
top11_overall_mean         float64
top11_overall_std          float64
top11_overall_min            int64
top11_overall_max            int64
top11_potential_mean       float64
top11_age_mean             float64
top11_height_mean          float64
top11_weight_mean          float64
top11_value_sum_eur        float64
top11_wage_sum_eur         float64
top11_pace_mean            float64
top11_shooting_mean        float64
top11_passing_mean         float64
top11_dribbling_mean       float64
top11_defending_mean       float64
top11_physic_mean   

In [27]:
np.sort(list(teams_strength['nationality_name']))

array(['Argentina', 'Australia', 'Austria', 'Belgium', 'Bolivia',
       'Brazil', 'Bulgaria', 'Cameroon', 'Canada', 'Chile', 'China',
       'Colombia', 'Croatia', 'Czechia', 'Denmark', 'Ecuador', 'Egypt',
       'England', 'Finland', 'France', 'Germany', 'Ghana', 'Greece',
       'Hungary', 'Iceland', 'India', 'Italy', 'Ivory_Coast',
       'Korea_Republic', 'Mexico', 'Morocco', 'Netherlands',
       'New_Zealand', 'Northern_Ireland', 'Norway', 'Paraguay', 'Peru',
       'Poland', 'Portugal', 'Qatar', 'Republic_of_Ireland', 'Romania',
       'Russia', 'Saudi_Arabia', 'Scotland', 'Slovenia', 'South_Africa',
       'Spain', 'Sweden', 'Switzerland', 'Turkey', 'Ukraine',
       'United_States', 'Uruguay', 'Venezuela', 'Wales'], dtype='<U19')